#### 两者结合（Megatron-DeepSpeed）通常是目前训练千亿参数以上模型的行业标准。
- Megatron 负责把模型“切得碎”（算得快），DeepSpeed 负责把显存“挤得干”（省得多），两者联手才能训好现在的千亿参数大模型。

| 特性 | Megatron-LM (NVIDIA) | DeepSpeed (Microsoft) |
| :--- | :--- | :--- |
| 核心强项 | 3D 并行策略 (张量并行 TP + 流水并行 PP + 数据并行 DP) | ZeRO 显存优化 (消除冗余) + 内核融合 |
| 主要解决的问题 | 模型太大，单卡/单节点放不下，必须把模型切开算。 | 显存不够用（即使模型能放下，优化器状态也占太多），想省显存或增大 Batch Size。 |
| 并行粒度 | 细粒度：在算子级别切分矩阵乘法（TP），在层级别切分模型（PP）。 | 粗粒度：主要在数据并行（DP）基础上，对优化器状态、梯度、参数进行分片。 |
| 代码侵入性 | 高：通常需要按照 Megatron 的特定格式重写模型代码。 | 低：只需几行代码即可集成到现有 PyTorch 模型中（尤其是 ZeRO）。 |
| 生态背景 | NVIDIA GPU 硬件深度优化，依赖 NCCL。 | 微软 Azure 云环境优化，支持多种硬件后端。 |



最佳实践方案：Megatron-DeepSpeed 混合架构
- 节点内 (Intra-node)：使用 Megatron 的 张量并行 (TP)。利用 NVLink 的高带宽，快速完成算子级切分计算。
- 节点间 (Inter-node)：使用 DeepSpeed 的 ZeRO (通常是 ZeRO-1 或 ZeRO-2) 或 Megatron 的 流水并行 (PP)。
- 具体组合：
    - TP (Megatron) 处理大矩阵计算。
    - PP (Megatron/DeepSpeed) 处理层间切分。
    - DP + ZeRO (DeepSpeed) 处理数据并行时的显存冗余。

| 场景 | 推荐方案 | 理由 |
| :--- | :--- | :--- |
| 中小模型 (<10B) | DeepSpeed (ZeRO-2/3) | 配置简单，无需改写模型代码，显存节省效果立竿见影，单卡/多卡微调首选。 |
| 超大模型 (>100B) 预训练 | Megatron-DeepSpeed | 必须利用 TP/PP 来保证计算效率，同时利用 ZeRO 来优化显存。这是目前工业界的标准。 |
| 推理 (Inference) | DeepSpeed-Inference 或 vLLM | DeepSpeed 提供了专门的推理引擎，支持动态批处理和算子融合；vLLM 则在 PagedAttention 上表现更佳。 |
| 科研/快速验证 | PyTorch FSDP | PyTorch 原生支持的 Fully Sharded Data Parallel，功能类似 DeepSpeed ZeRO-3，无需安装额外库，生态兼容性好。 |

